In [1]:
import numpy as np
import sys

from lib.envs.gridworld import GridworldEnv

env = GridworldEnv()


## 策略评估方法
def policy_eval(policy, env, discount_factor =1, threshold=0.00001):
	# 初始化各状态的状态值函数
	V = np.zeros(env.nS)
	i = 0

	while True:
		value_delta = 0
		# 遍历各状态
		for s in range(env.nS):
			v = 0

			for a, action_prob in enumerate(policy[s]):
				for prob, next_state, reward, done in env.P[s][a]:
					# v += action_prob * prob * (reward + discount_factor * V[next_state])
					v += action_prob * (reward + discount_factor * prob * V[next_state])
					
			value_delta = max(value_delta, np.abs(v - V[s]))
			V[s] =v
		


		i += 1
		if value_delta < threshold:
			break
	return np.array(V)


In [2]:
## 策略改进方法
v_num = 1
i_num = 1

# 根据传入的四个行为选择值函数最大的索引，返回的是一个索引数组和一个行为策略
def get_max_index(action_values):
	indexs = []
	policy_arr = np.zeros(len(action_values))
	action_max_value = np.max(action_values)

	for i in range(len(action_values)):
		action_value = action_values[i]

		if action_value == action_max_value:
			indexs.append(i)
			policy_arr[i] = 1
	return indexs, policy_arr

# 将策略中每行可能行为改为元组形式，方便对多个方向表示
def change_policy(policys):
	action_tuple = []

	for policy in policys:
		indexs, policy_arr = get_max_index(policy)
		action_tuple.append(tuple(indexs))
	return action_tuple	
	
# policy_improvement是策略改进方法，输入为环境env，策略评估函数policy_eval
# 输出为最优值函数和最优策略
def policy_improvement(env, policy_eval_fn = policy_eval, discount_factor=1.0):
	# 初始化一个随机策略
	policy = np.ones([env.nS, env.nA])/env.nA

	while True:
		global i_num
		global v_num

		v_num = 1

		# 评估当前的策略，输出为各状态当前的状态值函数
		V = policy_eval_fn(policy, env, discount_factor)
		#定义一个当前策略是否改变的标识
		policy_stable = True

		#遍历各状态
		for s in range(env.nS):
			# 取出当前状态下最优行为的索引值
			chosen_a = np.argmax(policy[s])
			
			# 初始化行为数组[0,0,0,0]
			action_values = np.zeros(env.nA)
			for a in range(env.nA):
				for prob, next_state, reward, done in env.P[s][a]:
					action_values[a] += prob * (reward + discount_factor * V[next_state])
			
			best_a_arr, policy_arr = get_max_index(action_values)
			if chosen_a not in best_a_arr:
				policy_stable = False
			policy[s] = policy_arr

		i_num += 1

		# 如果当前策略没有发生改变，即已经到了最优策略，返回
		if policy_stable:
			return policy, V
		
env = GridworldEnv()
policy, v = policy_improvement(env)
update_policy_type = change_policy(policy)	

In [3]:

print("Policy Probability Distribution:")
print(policy)



Policy Probability Distribution:
[[0. 1. 1. 0.]
 [0. 1. 1. 0.]
 [0. 1. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 1.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [1. 1. 1. 1.]
 [0. 0. 0. 1.]
 [1. 1. 0. 0.]
 [1. 1. 0. 0.]
 [1. 1. 0. 0.]
 [1. 0. 0. 0.]
 [1. 0. 0. 1.]
 [1. 1. 0. 0.]
 [1. 1. 0. 0.]
 [1. 1. 0. 0.]
 [1. 0. 0. 0.]
 [1. 0. 0. 1.]
 [1. 1. 0. 0.]
 [1. 1. 0. 0.]
 [1. 1. 0. 0.]
 [1. 0. 0. 0.]
 [1. 0. 0. 1.]]


In [4]:
print("Reshaped Grid Policy (0=up, 1=right, 2=down, 3=left):")
print(np.reshape(np.argmax(policy, axis=1), env.shape))
print("")



Reshaped Grid Policy (0=up, 1=right, 2=down, 3=left):
[[1 1 1 2 2]
 [1 1 1 0 3]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]]



In [5]:
print("Value Function:")
print(v)
print("")



Value Function:
[-4. -3. -2. -1. -2. -3. -2. -1.  0. -1. -4. -3. -2. -1. -2. -5. -4. -3.
 -2. -3. -6. -5. -4. -3. -4.]



In [6]:
print("Reshaped Grid Value Function:")
print(v.reshape(env.shape))
print("")

Reshaped Grid Value Function:
[[-4. -3. -2. -1. -2.]
 [-3. -2. -1.  0. -1.]
 [-4. -3. -2. -1. -2.]
 [-5. -4. -3. -2. -3.]
 [-6. -5. -4. -3. -4.]]

